# Unit 05 - Difference-in-Differences (Demo) · **V2 material**

**Atoms served:** `U05-A6` (**quasi-experimental methods at production scale** - Netflix case in reading; **no video** on DiD)

**Estimated runtime:** ~20 seconds

**After this notebook you can:** compare after-only and before-after estimates to DiD, plot parallel pre-trends, and see the estimate fail when that assumption breaks.

## Without code

1. True treatment effect: 5 units.
2. After-only gap: about **13** - wildly biased high, because treated cities started about 8 units higher.
3. Before-after on treated only: about **8** - biased by the common time trend.
4. DiD estimate: **5.0**, the true effect, when pre-trends are parallel.
5. Broken pre-trends: DiD drifts to about **4** - precise, and wrong.

**Note:** `V13` covers natural experiments but never names DiD. Netflix runs this at platform scale; this notebook is where you meet the method.

## 1. The question

A policy lands in some cities but not others - you did not assign treatment. Can you still estimate what changed - and what assumption makes the number trustworthy?

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

Four periods (two pre, two post) for treated and control units. A common time trend plus a true DiD effect of 5 on treated units after period 2.

Generate panel data with parallel pre-trends between groups.

In [ ]:
periods = np.array([0, 1, 2, 3])
true_effect = 5.0
n_units = 40

rows = []
for u in range(n_units):
    treated = 1 if u < n_units // 2 else 0
    unit_fe = np.random.normal(20 if treated else 12, 2)
    for t in periods:
        y = unit_fe + 1.5 * t  # common trend
        post = 1 if t >= 2 else 0
        if treated and post:
            y += true_effect
        rows.append({'unit': u, 'period': t, 'treated': treated, 'post': post, 'y': y})
panel = pd.DataFrame(rows)
print(panel.groupby(['treated','post'])['y'].mean().round(2))

## 4. The naive move

Compare treated vs control **after only**, or compare pre vs post **within treated only**. Both ignore part of the panel.

Compute both naive estimators alongside the true effect of 5.

In [ ]:
post_only = (panel.query('post==1 and treated==1')['y'].mean() -
               panel.query('post==1 and treated==0')['y'].mean())
treated_ba = (panel.query('treated==1 and post==1')['y'].mean() -
              panel.query('treated==1 and post==0')['y'].mean())
print('After-only estimate:', round(post_only, 2), '(true:', true_effect, ')')
print('Treated before-after:', round(treated_ba, 2), '(true:', true_effect, ')')

Both naive estimators miss the truth - one confounds group levels, the other confounds the time trend.

## 5. What actually happens

**Difference-in-differences.** Fit `y ~ treated + post + treated:post` (or equivalent). The interaction coefficient is the DiD estimator.

Compare the DiD estimate to the true effect of 5.

In [ ]:
panel['treated_post'] = panel['treated'] * panel['post']
did = smf.ols('y ~ treated + post + treated_post', data=panel).fit()
did_est = did.params['treated_post']
print('DiD estimate:', round(did_est, 2), '(true:', true_effect, ')')

With parallel pre-trends, DiD lands near the true effect. The assumption **is** the lesson.

**Parallel pre-trends.** Plot group means by period in the pre-period. Slopes should match.

In [ ]:
means = panel.groupby(['period', 'treated'])['y'].mean().unstack()
fig, ax = plt.subplots()
ax.plot(means.index, means[0], 'o-', label='control')
ax.plot(means.index, means[1], 's-', label='treated')
ax.axvline(1.5, linestyle='--')
ax.set_xlabel('period')
ax.set_ylabel('mean outcome')
ax.set_title('Parallel pre-trends support DiD')
ax.legend()
plt.show()

Before the dashed line, both groups rise together - that is what "parallel trends" means in a picture.

**Break the assumption.** Add an extra trend on treated units during pre-period only. DiD should drift away from 5.

In [ ]:
rows2 = []
for u in range(n_units):
    treated = 1 if u < n_units // 2 else 0
    unit_fe = np.random.normal(20 if treated else 12, 2)
    for t in periods:
        y = unit_fe + 1.5 * t
        if treated and t < 2:
            y += 2.0 * t  # diverging pre-trend on treated
        post = 1 if t >= 2 else 0
        if treated and post:
            y += true_effect
        rows2.append({'unit': u, 'period': t, 'treated': treated, 'post': post, 'y': y,
                      'treated_post': treated * post})
panel2 = pd.DataFrame(rows2)
did_bad = smf.ols('y ~ treated + post + treated_post', data=panel2).fit().params['treated_post']
print('DiD with broken pre-trends:', round(did_bad, 2), '(true:', true_effect, ')')

When pre-trends diverge, DiD is precise about the wrong thing. Plot before you publish.

## 6. What you do about it

- Reach for **DiD** when treatment was not randomly assigned but you have pre and post on treated and control (`U05-A6`).
- Always plot **pre-trends** before you trust the interaction term.
- Name rejected alternatives (after-only, simple before-after) in your write-up.

**When this matters less:** A clean randomised A/B test - use that instead.

---

**Takeaway:** DiD works when control units show you what treated units would have done without treatment. Parallel pre-trends is not a footnote - it is the whole claim.

**Back to the unit:** [V2 unit 05](../V2/units/unit-05-experiment-types/README.md)